# Validation 1D Analytique du Transport

Ce notebook valide la précision numérique du module de transport de `seapopym` en comparant les résultats de simulation avec une **solution analytique exacte**.

## Objectif
Prouver que le schéma numérique résout correctement l'équation d'advection-diffusion 1D :
$$ \frac{\partial C}{\partial t} + u \frac{\partial C}{\partial x} = D \frac{\partial^2 C}{\partial x^2} $$

## Méthodologie
1.  **Approximation 1D dans un modèle 2D** : On utilise un domaine "ruban" très étroit (3 points en Y) centré sur l'équateur. Les conditions sont uniformes en Y, ce qui annule les flux nord/sud.
2.  **Cas Test** : Transport d'une bouffée Gaussienne.
    - Vitesse $u$ constante (vers l'Est).
    - Diffusion $D$ constante.
3.  **Solution Analytique** : Une gaussienne convectée et diffusée :
    $$ C(x,t) = \frac{A}{\sqrt{4\pi(D t + \sigma_0^2)}} \exp\left( - \frac{(x - x_0 - ut)^2}{4(D t + \sigma_0^2)} \right) $$
4.  **Métriques** : Comparaison visuelle et calcul de l'erreur L2 et L-infini.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from seapopym.standard.coordinates import Coordinates
from seapopym.transport import (
    compute_spherical_cell_areas,
    compute_spherical_dx,
    compute_spherical_dy,
    compute_spherical_face_areas_ew,
    compute_spherical_face_areas_ns,
    compute_transport_numba,
)

print("✅ Imports réussis")

## 1. Définition de la Solution Analytique

In [ ]:
def gaussian_analytic(x, t, x0, u, D, sigma0, mass_total):
    """
    Solution analytique 1D de l'équation d'advection-diffusion pour une gaussienne.

    Args:
        x: Grille spatiale (m)
        t: Temps (s)
        x0: Position initiale du centre (m)
        u: Vitesse d'advection (m/s)
        D: Coefficient de diffusion (m²/s)
        sigma0: Écart-type initial (m)
        mass_total: Masse totale conservée (intégrale)

    Returns:
        Concentration C(x,t)
    """
    # Variance au temps t : sigma^2 = 2*D*t + sigma0^2
    # Note: En physique standard, variance = 2Dt. Ici on ajoute la variance initiale.
    curr_variance = 2 * D * t + sigma0**2
    curr_sigma = np.sqrt(curr_variance)

    # Centre de la gaussienne au temps t
    curr_x0 = x0 + u * t

    # Facteur de normalisation pour conserver la masse
    # L'intégrale de exp(-x^2/(2s^2)) est s * sqrt(2*pi)
    # On veut que Int(C) = mass_total
    # Donc A * sigma * sqrt(2*pi) = mass_total => A = mass_total / (sigma * sqrt(2*pi))
    normalization = mass_total / (curr_sigma * np.sqrt(2 * np.pi))

    # Gaussienne : N * exp( - (x-mu)^2 / (2*sigma^2) )
    c = normalization * np.exp(-((x - curr_x0) ** 2) / (2 * curr_variance))
    return c


print("✅ Fonction analytique définie")

## 2. Configuration de la Simulation (Approximation 1D)

In [ ]:
# Paramètres de la grille
nx = 1000
ny = 3  # Minimal pour calculer les dérivees, centré sur l'équateur
dlon_deg = 0.1  # Résolution de 0.1 degré (~11km à l'équateur)
dlat_deg = 0.1

# Centré sur l'équateur et Greenwhich
lons_deg = np.linspace(0, nx * dlon_deg, nx)
lats_deg = np.linspace(-dlat_deg, dlat_deg, ny)

lats = xr.DataArray(lats_deg, dims=[Coordinates.Y.value])
lons = xr.DataArray(lons_deg, dims=[Coordinates.X.value])

# Calcul des métriques géométriques
cell_areas = compute_spherical_cell_areas(lats, lons)
dx = compute_spherical_dx(lats, lons)
dy = compute_spherical_dy(lats, lons)
face_areas_ew = compute_spherical_face_areas_ew(lats, lons)
face_areas_ns = compute_spherical_face_areas_ns(lats, lons)

# Coordonnées en mètres (approximées pour l'analytique, x = distance le long de l'équateur)
# On prend la ligne centrale (index 1)
dx_mean = dx.isel({Coordinates.Y.value: 1}).mean().values
x_meters = lons.values * 111320.0  # Approx rapide 1 deg ~ 111.32 km à l'équateur
# Ou plus précis : cumul des dx
x_meters = np.cumsum(np.full(nx, dx_mean)) - dx_mean / 2  # Centré aux cellules

print(f"Grille : {nx} x {ny}")
print(f"Resolution X ~ {dx_mean / 1000:.2f} km")
print(f"Longueur domaine ~ {x_meters[-1] / 1000:.1f} km")

In [ ]:
# Paramètres Physiques
u_magnitude = 0.5  # m/s
D_coeff = 2500.0  # m²/s

# Champs 2D uniformes
u_da = xr.DataArray(np.full((ny, nx), u_magnitude), dims=[Coordinates.Y.value, Coordinates.X.value])
v_da = xr.DataArray(np.zeros((ny, nx)), dims=[Coordinates.Y.value, Coordinates.X.value])
D_da = xr.DataArray(np.full((ny, nx), D_coeff), dims=[Coordinates.Y.value, Coordinates.X.value])
mask = xr.DataArray(np.ones((ny, nx)), dims=[Coordinates.Y.value, Coordinates.X.value])

# Paramètres temporels
# Contraintes CFL pour la stabilité
cfl_adv_target = 0.5  # CFL pour l'advection: u * dt / dx < 1
cfl_diff_target = 0.5  # CFL pour la diffusion: D * dt / dx² < 0.5

# Calcul des pas de temps limites
dt_advection = cfl_adv_target * dx_mean / u_magnitude
dt_diffusion = cfl_diff_target * dx_mean**2 / D_coeff

# Prendre le minimum pour respecter les deux contraintes
dt = min(dt_advection, dt_diffusion)
dt = float(int(dt))  # Arrondi

# Calcul des CFL effectifs
cfl_adv_effective = u_magnitude * dt / dx_mean
cfl_diff_effective = D_coeff * dt / dx_mean**2

print("=== Contraintes CFL ===")
print(f"dt limite (advection) : {dt_advection:.1f} s")
print(f"dt limite (diffusion) : {dt_diffusion:.1f} s")
print(f"\nPas de temps choisi (dt) : {dt} s")
print(f"\nCFL effectifs:")
print(f"  - Advection : {cfl_adv_effective:.4f} (< 1.0 requis)")
print(f"  - Diffusion : {cfl_diff_effective:.4f} (< 0.5 requis)")

if dt == int(dt_advection):
    print("\n⚠️  Contrainte limitante : ADVECTION")
else:
    print("\n⚠️  Contrainte limitante : DIFFUSION")

duration_hours = 24 * 7
total_time = duration_hours * 3600
n_steps = int(total_time / dt)
print(f"\nNombre de pas de temps : {n_steps}")

## 3. Initialisation

In [ ]:
# Conditions Initiales (Gaussienne)
x0_center = x_meters[nx // 4]  # Start at 1/4 of domain
sigma0 = 80000.0  # 20 km thickness
target_mass = 1.0

# Calcul analytique au temps t=0 pour initialiser
c_init_1d = gaussian_analytic(x_meters, 0, x0_center, u_magnitude, D_coeff, sigma0, target_mass)

# En 2D (kg/m^2 ou concentration)
# Attention: notre analytique est une densité linéique (1/m) ou massique ?
# L'équation de transport conserve la quantité (Concentration * Volume) ou (Densité * Aire)
# Ici on transporte une "densité linéique" C(x). Transformons ça en densité surfacique uniforme sur Y.
# Si C_1d est en kg/m (intégrale = kg), alors Densité 2D = C_1d / Ly.
# Simplification : On travaille directement en valeurs arbitraires, ce qui compte c'est la forme.
# On initialise la grille 2D avec le profil 1D.

biomass_init_vals = np.tile(c_init_1d, (ny, 1))
biomass_init = xr.DataArray(biomass_init_vals, dims=[Coordinates.Y.value, Coordinates.X.value])

# Normalisation exacte de la masse pour être propre
current_mass = (biomass_init * cell_areas).sum().values
biomass_init = biomass_init * (target_mass / current_mass)

print(f"Masse initiale (simulation) : {(biomass_init * cell_areas).sum().values:.4f}")

# Plot initial
plt.plot(
    x_meters / 1000, biomass_init.isel({Coordinates.Y.value: 1}), label="Initial Condition (t=0)"
)
plt.xlabel("Distance (km)")
plt.ylabel("Concentration")
plt.legend()
plt.show()

## 4. Exécution de la Simulation

In [ ]:
biomass = biomass_init.copy()

for step in range(n_steps):
    result = compute_transport_numba(
        state=biomass,
        u=u_da,
        v=v_da,
        D=D_da,
        dx=dx,
        dy=dy,
        cell_areas=cell_areas,
        face_areas_ew=face_areas_ew,
        face_areas_ns=face_areas_ns,
        boundary_north=0,  # Closed
        boundary_south=0,
        boundary_east=0,
        boundary_west=0,
        mask=mask,
    )

    tendency = result["advection_rate"] + result["diffusion_rate"]
    biomass = biomass + tendency * dt

# Résultat de la ligne centrale
biomass_final_sim = biomass.isel({Coordinates.Y.value: 1}).values

## 5. Comparaison avec l'Analytique

In [ ]:
# Calcul de la solution analytique finale
# Note: On re-normalise l'analytique 1D pour qu'elle ait la même intégrale 'surface' que la simulation
# Pour comparaison purement de forme.
mass_final_sim = (biomass * cell_areas).sum().values

# On génère la forme 1D théorique
shape_final_analytic = gaussian_analytic(
    x_meters, total_time, x0_center, u_magnitude, D_coeff, sigma0, mass_total=1.0
)

# On la projette sur la grille 2D (virtuellement) pour avoir la même échelle
# Area_factor moyen d'une cellule
avg_cell_area = cell_areas.isel({Coordinates.Y.value: 1}).values
# La masse totale 'target_mass=1.0' dans gaussian_analytic correspond à l'intégrale C(x) dx.
# Dans la simu, Masse = Somme(C_ij * Area_ij).
# Si C_sim est proche de C_analytic, alors C_sim \approx shape_final_analytic * Factor
# Le plus simple : on normalise les deux courbes par leur maximum ou leur aire pour comparer la forme.
# Comparons les profils bruts en s'assurant que l'intégrale correspond.

# Normalisation par l'intégrale pour comparaison distribution de probabilité
sim_integral = np.trapz(biomass_final_sim, x_meters)
analytic_integral = np.trapz(shape_final_analytic, x_meters)

biomass_final_sim_norm = biomass_final_sim / sim_integral
shape_final_analytic_norm = shape_final_analytic / analytic_integral

# Visualisation
plt.figure(figsize=(10, 6))
plt.plot(
    x_meters / 1000,
    biomass_final_sim_norm,
    "o",
    label="Simulation (Numérique)",
    markersize=4,
    alpha=0.7,
)
plt.plot(x_meters / 1000, shape_final_analytic_norm, "r-", label="Analytique (Exact)", linewidth=2)
plt.title(f"Transport 1D après {duration_hours}h : Advection + Diffusion")
plt.xlabel("Position X (km)")
plt.ylabel("Densité de probabilité")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Zoom sur le pic
plt.figure(figsize=(10, 6))
plt.plot(x_meters / 1000, biomass_final_sim_norm, "o-", label="Simulation", markersize=4)
plt.plot(x_meters / 1000, shape_final_analytic_norm, "r--", label="Analytique")
plt.xlim(
    (x0_center + u_magnitude * total_time) / 1000 - 100,
    (x0_center + u_magnitude * total_time) / 1000 + 100,
)
plt.title("Zoom sur le pic")
plt.xlabel("Position X (km)")
plt.grid(True)
plt.legend()
plt.show()

## 6. Analyse de l'Erreur

In [ ]:
# Calcul des erreurs
error = biomass_final_sim_norm - shape_final_analytic_norm
l2_error = np.sqrt(np.mean(error**2))
linf_error = np.max(np.abs(error))

print(f"Erreur L2 (RMSE) : {l2_error:.2e}")
print(f"Erreur L-infinie (Max Abs) : {linf_error:.2e}")

# Critère de validation
# On s'attend à une erreur faible (< 1% du pic max) pour un schéma décent
max_val = np.max(shape_final_analytic_norm)
relative_error_percent = (linf_error / max_val) * 100

print(f"Erreur relative au pic : {relative_error_percent:.2f}%")

if relative_error_percent < 5.0:
    print("✅ VALIDATION RÉUSSIE : L'erreur numérique est inférieure à 5%")
else:
    print("⚠️ ATTENTION : L'erreur numérique semble élevée")